In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [ ]:
from pydantic import BaseModel, Field

# === 샘플 문서 준비 ===
document = """Adapterz(어댑터즈)는 스타트업코드에서 운영하는 개발 교재 서빙 서비스입니다.
어댑터즈의 모든 교재는 5단 분석법이라는 독자적인 교수법으로 작성됩니다.
5단 분석법은 기술 용어를 일반 명사와 고유 명사로 나누어 설명하고,
사용 이유와 방법, 다른 기술과의 비교까지 체계적으로 정리하는 방식입니다.
어댑터즈는 인공지능, 데이터 분석, 웹 개발, 인프라 분야의 교재를 제공합니다."""

# === 출력 스키마 정의 ===
# Structured Output, JsonOutputParser 페이지에서 사용한 것과 동일한 구조입니다.
class DocumentAnalysis(BaseModel):
    title: str = Field(description="문서의 제목")
    summary: str = Field(description="문서의 핵심 내용을 2~3문장으로 요약")
    keywords: list[str] = Field(description="문서의 핵심 키워드 3~5개")

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser

# === 파서 생성 ===
parser = PydanticOutputParser(pydantic_object=DocumentAnalysis)

# === format_instructions 확인 ===
print(parser.get_format_instructions())

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

# === Chat Model 생성 ===
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY
)

# === 프롬프트 구성 ===
prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 문서를 분석하세요.\n\n{format_instructions}"),
    ("human", "{document}")
])

# === LCEL 체인 구성 ===
chain = prompt | model | parser

# === 실행 ===
result = chain.invoke({
    "document": document,
    "format_instructions": parser.get_format_instructions()
})

# === 결과 확인 ===
print(f"타입: {type(result).__name__}")
print(f"제목: {result.title}")
print(f"요약: {result.summary}")
print(f"키워드: {result.keywords}")